# Qwen3-1.7B safety post-training on one Kaggle T4
This notebook only orchestrates repository scripts. Select conditions/seeds deliberately; it never launches the full matrix automatically.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-c', "import torch; print('torch', torch.__version__, 'CUDA', torch.version.cuda); print('GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')"], check=True)

## Repository and dependencies
Set `REPO` to the writable repository copy under `/kaggle/working`. Do not reinstall PyTorch/CUDA.

In [ ]:
from pathlib import Path
REPO = Path('/kaggle/working/black-box-diff-exploration')
assert (REPO / 'train.py').exists(), 'Copy/clone the repository into /kaggle/working and update REPO'
%cd {REPO}
%env HF_HOME=/kaggle/working/hf_cache

In [ ]:
!pip install -q -r requirements-kaggle.txt
!python -c "import torch, transformers, peft, trl; print('final:', torch.__version__, transformers.__version__, peft.__version__, trl.__version__)"

## Preflight, frozen data validation, and balance

In [ ]:
!python scripts/kaggle_preflight.py
!python scripts/freeze_evaluation.py
!python scripts/validate_experiment.py
!python scripts/check_experimental_balance.py --model-tokenizer

## Smoke test
Synthetic fixtures are written only below the `_smoke` artifact namespace and are not experimental data.

In [ ]:
!python run_training_matrix.py --conditions M1 M2 M3 --seeds 42 --smoke-test

## Selected training job
Change these deliberately. Generate and review cached M3 targets before selecting M3.

In [ ]:
CONDITION = 'M2'
SEED = 42
CONFIGS = {'M1': 'configs/m1_benign.yaml', 'M2': 'configs/m2_safety_sft.yaml', 'M3': 'configs/m3_constitutional.yaml'}
assert CONDITION in CONFIGS and SEED in (42, 123, 456)
!python train.py --config {CONFIGS[CONDITION]} --seed {SEED} --resume

## Immediate frozen audit
The built-in heuristic verifies the pipeline only. Configure a validated `SafetyScorer` for research conclusions.

In [ ]:
!python evaluate_safety.py --condition {CONDITION} --seed {SEED}

## Artifact summary and optional commands

In [ ]:
!python run_training_matrix.py --conditions M1 M2 M3 --seeds 42 123 456 --dry-run
!find /kaggle/working/artifacts -maxdepth 4 -type f | sort | tail -100
# Deliberately run the full matrix only when intended:
# !python run_training_matrix.py --conditions M1 M2 M3 --seeds 42 123 456 --resume
# Optional M4 after its matching M3:
# !python train_dpo.py --config configs/m4_dpo.yaml --seed 42 --resume